In [8]:
from pathlib import Path
import os
import csv


In [10]:
# split_sidd.py 位于 graduation-design/src/
src_dir = os.getcwd()
src_dir = Path(src_dir)
project_root = src_dir.parent

sidd_root = (
    project_root
    / "data"
    / "SIDD_Medium_Srgb"
    / "Data"
)

output_dir = project_root / "data" / "splits"
output_dir.mkdir(parents=True, exist_ok=True)

output_csv = output_dir / "sidd_split.csv"

print("SIDD目录：", sidd_root)
print("输出文件：", output_csv)

SIDD目录： d:\CodeRepository\graduation‑design\data\SIDD_Medium_Srgb\Data
输出文件： d:\CodeRepository\graduation‑design\data\splits\sidd_split.csv


In [19]:
TRAIN_SCENES = {
    "001", "002", "003",
    "004", "005", "006"
}

VAL_SCENES = {
    "007", "010"
}

TEST_SCENES = {
    "009", "008"
}

In [20]:
scene_dirs = sorted(
    path
    for path in sidd_root.iterdir()
    if path.is_dir()
)

print("Scene Instance数量：", len(scene_dirs))

Scene Instance数量： 160


In [21]:
def get_split(scene_id):

    if scene_id in TRAIN_SCENES:
        return "train"

    if scene_id in VAL_SCENES:
        return "val"

    if scene_id in TEST_SCENES:
        return "test"

    raise ValueError(f"未知scene：{scene_id}")

In [22]:
records = []

for scene_dir in scene_dirs:

    # 例如：
    # 0001_001_S6_00100_00060_3200_L

    parts = scene_dir.name.split("_")

    scene_instance = parts[0]
    scene_id = parts[1]
    smartphone = parts[2]
    iso = parts[3]
    shutter = parts[4]
    color_temp = parts[5]
    brightness = parts[6]

    split = get_split(scene_id)

    noisy_files = sorted(
        scene_dir.glob("*_NOISY_SRGB_*.PNG")
    )

    for noisy_path in noisy_files:

        gt_name = noisy_path.name.replace(
            "_NOISY_SRGB_",
            "_GT_SRGB_"
        )

        gt_path = scene_dir / gt_name

        if not gt_path.exists():
            raise FileNotFoundError(
                f"找不到对应GT：{noisy_path}"
            )

        records.append({
            "split": split,
            "scene_id": scene_id,
            "scene_instance": scene_instance,
            "smartphone": smartphone,
            "iso": iso,
            "shutter": shutter,
            "color_temp": color_temp,
            "brightness": brightness,

            # 保存相对于项目根目录的路径
            "noisy_path": str(
                noisy_path.relative_to(project_root)
            ),

            "gt_path": str(
                gt_path.relative_to(project_root)
            ),
        })

In [23]:
fieldnames = [
    "split",
    "scene_id",
    "scene_instance",
    "smartphone",
    "iso",
    "shutter",
    "color_temp",
    "brightness",
    "noisy_path",
    "gt_path",
]

with output_csv.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(records)

print("划分文件已生成：", output_csv)

划分文件已生成： d:\CodeRepository\graduation‑design\data\splits\sidd_split.csv


In [24]:
train_records = [
    r for r in records
    if r["split"] == "train"
]

val_records = [
    r for r in records
    if r["split"] == "val"
]

test_records = [
    r for r in records
    if r["split"] == "test"
]

print()
print("=" * 40)
print("SIDD 数据划分结果")
print("=" * 40)

print("总配对数量：", len(records))
print("Train：", len(train_records))
print("Validation：", len(val_records))
print("Test：", len(test_records))


SIDD 数据划分结果
总配对数量： 320
Train： 222
Validation： 46
Test： 52


In [25]:
train_scene_ids = {
    r["scene_id"]
    for r in train_records
}

val_scene_ids = {
    r["scene_id"]
    for r in val_records
}

test_scene_ids = {
    r["scene_id"]
    for r in test_records
}

assert train_scene_ids.isdisjoint(val_scene_ids)
assert train_scene_ids.isdisjoint(test_scene_ids)
assert val_scene_ids.isdisjoint(test_scene_ids)

print()
print("Train scenes：", sorted(train_scene_ids))
print("Val scenes：", sorted(val_scene_ids))
print("Test scenes：", sorted(test_scene_ids))

print()
print("场景泄漏检查通过")


Train scenes： ['001', '002', '003', '004', '005', '006']
Val scenes： ['007', '010']
Test scenes： ['008', '009']

场景泄漏检查通过
